# Test clip pre-labels
A COCO-pretrained Faster R-CNN ("sports ball", not trained on SoccerNet) suggests a ball position per frame.
The suggestions are only a starting point: every 5th frame is checked and corrected by hand with `annotate_ball.py`.

In [ ]:
import numpy as np, cv2, torch, pandas as pd, matplotlib.pyplot as plt
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from common import read_video

T = 0.3  # min score
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
w = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
BALL = w.meta['categories'].index('sports ball')
model = fasterrcnn_resnet50_fpn(weights=w, min_size=1080, max_size=1920, box_score_thresh=0.05).to(dev).eval()  # 3x upscale

In [ ]:
frames, _ = read_video()
rows, scores = [], []
with torch.no_grad(), torch.autocast(dev, enabled=dev == 'cuda'):
    for i in range(0, len(frames), 4):
        x = [torch.from_numpy(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().div(255).to(dev) for f in frames[i:i + 4]]
        for j, o in enumerate(model(x)):
            k = o['labels'] == BALL
            b, s = o['boxes'][k].float().cpu().numpy(), o['scores'][k].float().cpu().numpy()
            scores.append(s[0] if len(s) else 0.0)
            if len(s) and s[0] >= T:
                rows.append((i + j, (b[0, 0] + b[0, 2]) / 2, (b[0, 1] + b[0, 3]) / 2))
pd.DataFrame(rows, columns=['frame', 'x', 'y']).round(1).to_csv('data/annotations/prelabels.csv', index=False)
print(f'ball suggested in {len(rows)}/{len(frames)} frames')

plt.figure(figsize=(14, 3))
plt.plot(scores)
plt.axhline(T, c='r')
plt.xlabel('frame')
plt.ylabel('ball score')

## Manual annotation
Run from the project folder: `python annotate_ball.py`

Every 5th frame is shown with the suggestion (yellow). Click the ball to correct it, ENTER to accept, N if no ball is visible.
Result: `data/annotations/denmark_wales_points.csv` (frame, visible, x, y).